# Hierarchical Reconciliation Comparison (Enhanced)

This notebook compares the proposed **GS-GLS** estimator against standard baselines (**OLS**, **MinT Sample**, **MinT Shrinkage**) on a challenging synthetic hierarchical dataset.

## Task Difficulty
We increase the difficulty compared to standard demos:
- **Larger Hierarchy**: Random tree with ~50 nodes.
- **Bias**: Base forecasts have systematic bias (simulating structural error).
- **Heteroscedasticity**: Noise variance increases over time.
- **Limited Data**: Training T=600 relative to full dimension (N*T_block = 490).

We evaluate performance in terms of Accuracy (MSE/MAE), Structural Coherence (Max Constraint Violation), and Efficiency.

In [1]:
import numpy as np
import pandas as pd
import time
import random
import matplotlib.pyplot as plt

from hierarchy import Hierarchy
from data_generator import HierarchicalDataGenerator
from gs_gls import GSGLS
import baselines

%load_ext autoreload
%autoreload 2

## 1. Setup and Data Generation

In [4]:
# Configuration: Enhanced Data
n_timesteps_train = 1000
n_timesteps_test = 200
n_t_block = 100

# 1. Random Hierarchy Generation
def generate_random_hierarchy(depth=4, branching_factor=3):
    structure = {}
    current_layer = ['Total']
    node_ctr = 1
    for d in range(depth):
        next_layer = []
        for parent in current_layer:
            n_children = random.randint(2, branching_factor)
            children = []
            for _ in range(n_children):
                child_name = f'Node_{d+1}_{node_ctr}'
                children.append(child_name)
                node_ctr += 1
            structure[parent] = children
            next_layer.extend(children)
        current_layer = next_layer
    return structure

structure = generate_random_hierarchy()
h = Hierarchy(structure)
n_nodes = h.n_nodes
print(f"Nodes: {n_nodes} (Bottom: {h.m_bottom})")

# 2. Generate Complex Data (Bias + Heteroscedasticity)
gen = HierarchicalDataGenerator(h, n_timesteps=n_timesteps_train + n_timesteps_test)
Y_true_all = gen.generate_ground_truth()
noise_all = gen.generate_spatiotemporal_noise(
    spatial_rho=1.5, 
    temporal_ar_coefs=[0.6, 0.2], 
    noise_scale=2.0,
    bias_scale=3.0,
    heteroscedastic=True
)
Y_hat_all = Y_true_all + noise_all

# Split Train/Test
residuals_train = noise_all[:, :n_timesteps_train]  # (n_nodes x T_train)
Y_hat_test = Y_hat_all[:, n_timesteps_train:]       # (n_nodes x T_test)
Y_true_test = Y_true_all[:, n_timesteps_train:]

print(f"Train Residuals Shape: {residuals_train.shape}")
print(f"Test Forecast Shape: {Y_hat_test.shape}")

Nodes: 55 (Bottom: 33)
Train Residuals Shape: (55, 1000)
Test Forecast Shape: (55, 200)


## 2. Preparation for Baselines

In [5]:
# Construct Global S matrix
S_sp = h.get_summing_matrix()
S_total = baselines.build_spatiotemporal_s(S_sp, n_t_block)
print(f"Global Summing Matrix S shape: {S_total.shape}")

# Prepare Sample Residuals for MinT
train_blocks = []
for i in range(0, residuals_train.shape[1] - n_t_block + 1, n_t_block):
    block = residuals_train[:, i:i+n_t_block]
    flat_block = block.flatten(order='F')
    train_blocks.append(flat_block)

residuals_samples = np.array(train_blocks) # (n_samples, N_total)
print(f"MinT Training Samples: {residuals_samples.shape}")
if residuals_samples.shape[1] > residuals_samples.shape[0]:
    print("WARNING: N > T. MinT (Sample) will be theoretically singular.")

Global Summing Matrix S shape: (5500, 3300)
MinT Training Samples: (10, 5500)


## 3. Evaluation Loop

In [6]:
results = []

# Pre-train GS-GLS
start_t = time.time()
gs_estimator = GSGLS(h)
gs_estimator.fit(residuals_train)
gs_gls_fit_time = time.time() - start_t
print(f"GS-GLS Fitted in {gs_gls_fit_time:.4f}s")

metrics = {'Method': [], 'MSE': [], 'MAE': [], 'Max_Incoherence': [], 'Time': []}

def eval_method(name, func, needs_train_data=False):
    mse_list = []
    mae_list = []
    incoh_list = []
    t_list = []
    
    # Loop over test blocks
    for i in range(0, Y_hat_test.shape[1] - n_t_block + 1, n_t_block):
        y_hat_block = Y_hat_test[:, i:i+n_t_block]
        y_true_block = Y_true_test[:, i:i+n_t_block]
        y_hat_flat = y_hat_block.flatten(order='F')
        
        start_time = time.time()
        
        try:
            if name == 'GS-GLS':
                y_tilde_block = gs_estimator.reconcile(y_hat_block)
                
            else:
                # Baselines
                if needs_train_data:
                    y_tilde_flat = func(y_hat_flat, residuals_samples, S_total)
                else:
                    y_tilde_flat = func(y_hat_flat, S_total)
                
                y_tilde_block = y_tilde_flat.reshape((n_nodes, n_t_block), order='F')
            
            end_time = time.time()
            
            # Compute Metrics
            mse = np.mean((y_tilde_block - y_true_block)**2)
            mae = np.mean(np.abs(y_tilde_block - y_true_block))
            
            # Incoherence: Check Full Structure
            max_incoh = 0.0
            for parent, children in h.structure.items():
                p_idx = h.node_to_idx[parent]
                c_indices = [h.node_to_idx[c] for c in children]
                agg = np.sum(y_tilde_block[c_indices, :], axis=0)
                curr_err = np.max(np.abs(y_tilde_block[p_idx, :] - agg))
                if curr_err > max_incoh: max_incoh = curr_err
            
            mse_list.append(mse)
            mae_list.append(mae)
            incoh_list.append(max_incoh)
            t_list.append(end_time - start_time)
            
        except np.linalg.LinAlgError:
            continue # Skip singular blocks for Sample MinT
        except Exception as e:
            print(f"Error: {e}")
            break
            
    if not mse_list: return np.nan, np.nan, np.nan, np.nan
    return np.mean(mse_list), np.mean(mae_list), np.mean(incoh_list), np.mean(t_list)

# Run Comparisons
print("Running OLS...")
m, m_ae, inc, t = eval_method('OLS (Identity)', baselines.ols_identity)
metrics['Method'].append('OLS (Identity)')
metrics['MSE'].append(m); metrics['MAE'].append(m_ae); metrics['Max_Incoherence'].append(inc); metrics['Time'].append(t)

print("Running MinT (Sample)...")
m, m_ae, inc, t = eval_method('MinT (Sample)', baselines.mint_sample, needs_train_data=True)
metrics['Method'].append('MinT (Sample)')
metrics['MSE'].append(m); metrics['MAE'].append(m_ae); metrics['Max_Incoherence'].append(inc); metrics['Time'].append(t)

print("Running MinT (Shrinkage)...")
m, m_ae, inc, t = eval_method('MinT (Shrinkage)', baselines.mint_shrink, needs_train_data=True)
metrics['Method'].append('MinT (Shrinkage)')
metrics['MSE'].append(m); metrics['MAE'].append(m_ae); metrics['Max_Incoherence'].append(inc); metrics['Time'].append(t)

print("Running GS-GLS...")
m, m_ae, inc, t = eval_method('GS-GLS', None)
metrics['Method'].append('GS-GLS (Proposed)')
metrics['MSE'].append(m); metrics['MAE'].append(m_ae); metrics['Max_Incoherence'].append(inc); metrics['Time'].append(t)

df_res = pd.DataFrame(metrics)
df_res

GS-GLS Fitted in 0.5997s
Running OLS...
Running MinT (Sample)...
Running MinT (Shrinkage)...
Running GS-GLS...


,Method,MSE,MAE,Max_Incoherence,Time
0,OLS (Identity),101.569225,7.898519,1.818989e-12,9.590847
1,MinT (Sample),100.683128,7.879893,1.864464e-11,6.491181
2,MinT (Shrinkage),106.457836,8.088402,9.094947e-12,4.363872
3,GS-GLS (Proposed),82.792620,7.210467,2.728484e-12,0.000000


## 4. Combined Complexity and Performance Table

We merge the empirical results with the theoretical complexity bounds.

In [7]:
# Theoretical Data
complexity_data = {
    'Method': ['OLS (Identity)', 'MinT (Sample)', 'MinT (Shrinkage)', 'GS-GLS (Proposed)'],
    'Covariance Estimation': ['N/A', '$O(N^2 T)$', '$O(N^2 T)$', '$O(N T \log T)$'],
    'Inversion/Projection': ['$O(N)$ (sparse)', '$O(N^3)$', '$O(N^3)$', '$O(N \log N + T \log T)$'],
    'Storage': ['$O(N)$', '$O(N^2)$', '$O(N^2)$', '$O(N)$']
}
df_theory = pd.DataFrame(complexity_data)

# Merge
df_final = pd.merge(df_theory, df_res, on='Method')

# Format
df_final['MSE'] = df_final['MSE'].map('{:.4f}'.format)
df_final['MAE'] = df_final['MAE'].map('{:.4f}'.format)
df_final['Time'] = df_final['Time'].map('{:.2e}'.format)
df_final['Max_Incoherence'] = df_final['Max_Incoherence'].map('{:.2e}'.format)

df_final

,Method,Covariance Estimation,Inversion/Projection,Storage,MSE,MAE,Max_Incoherence,Time
0,OLS (Identity),N/A,$O(N)$ (sparse),$O(N)$,101.5692,7.8985,1.82e-12,9.59e+00
1,MinT (Sample),$O(N^2 T)$,$O(N^3)$,$O(N^2)$,100.6831,7.8799,1.86e-11,6.49e+00
2,MinT (Shrinkage),$O(N^2 T)$,$O(N^3)$,$O(N^2)$,106.4578,8.0884,9.09e-12,4.36e+00
3,GS-GLS (Proposed),$O(N T \log T)$,$O(N \log N + T \log T)$,$O(N)$,82.7926,7.2105,2.73e-12,0.00e+00
